# 인공위성 영상 기반 산림 유형 분류 — 중간 공유**주제**인공위성 영상 데이터로 산림 유형을 분류하는 AI 모델 개발.LCA 산정에 필요한 산림 데이터로 활용 가능한지 평가.LCA 기준에서 탄소축적량이 쓰일 수 있다고 생각하고 아래 실험을 진행.---> **요약**> 분류 성능(F1)은 나오지만, LCA 산정에 필요한(?) **탄소량 오차가 큼**

## 1. 무엇을 했나위성 영상으로 산림 유형을 분류함 (침엽수림 / 활엽수림 / 혼효림).분류 결과에 탄소계수를 곱해서 탄소축적량을 구함.

## 2. 모델 만든 과정**데이터**| | 내용 ||---|---|| 정답 데이터 | 1:5,000 임상도. 산림청 제공 || 입력 | Google Earth Engine에서 Sentinel-2. 10m 해상도, 무료 |임상도 경계를 그대로 위성 촬영 범위로 설정.**모델 설정**| | 내용 ||---|---|| feature 38개 | 여름 10 + 낙엽기 10 + 식생지수 + 주변 픽셀 || 분류기 | RandomForest (딥러닝 아님. 가져옴) || 학습량 | 모든 조건에서 20만 픽셀 || 라벨 | 조사 갱신년도 2024 이후만 사용 (2020 이후도 쓸 수 있지만 2024로 통일) |

## 3. 학습 결과대전 지역으로 학습했는데 **F1 0.628 / 탄소 오차 +0.1%.** 큰 문제 없음.

## 4. 타일 분할픽셀을 랜덤하게 나누면 학습·평가 타일이 인접할 가능성 발생→ **500픽셀 타일 단위로 묶음**타일도 동일한 문제가 발생할 수 있음→ **동서 분리****결과: F1 0.572 / 탄소 오차 +0.77%**같은 지역, 같은 데이터, 같은 모델인데 **평가 영역이 분리되니 오차가 10배 이상.**→ 그럼 지역이 바뀌면?

## 5. 지역 변경 — 대전 · 홍천 · 순천| 학습 → 평가 | F1 | 탄소 오차 ||---|---|---|| 대전 → 대전 | 0.624 | −1.05% || 대전 → 홍천 | 0.569 | **+2.73%** || 대전 → 순천 | 0.550 | −1.57% || 홍천 → 홍천 | 0.625 | +0.12% || 홍천 → 대전 | 0.581 | −1.51% || 홍천 → 순천 | 0.539 | −3.48% || 순천 → 순천 | 0.630 | +0.84% || 순천 → 대전 | 0.583 | +3.12% || **순천 → 홍천** | 0.484 | **+5.92%** |*(3절은 초기 설정·임시 탄소계수 기준이라 대전→대전 값이 이 표와 다름)*

## 6. F1보다 탄소량 오차가 큼| | 지역 내 → 지역 간 ||---|---|| F1 | 0.626 → 0.551 || 잘못 분류한 픽셀 | **1.25배** || **탄소량** | **0.12% → 5.92%** |**F1만으로 탄소 오차를 측정하기 어려움.** 오차의 부호도 다름.| 학습 지역 | F1 | 탄소 오차 ||---|---|---|| 홍천 → 대전 | 0.581 | **−1.51%** || 순천 → 대전 | 0.583 | **+3.12%** |같은 평가 지역, 사실상 같은 F1인데 **부호가 반대.**그리고 총량은 상쇄된 값임. **타일 간 표준편차까지 고려해야 함.**(표준편차는 있지만 어느 정도까지가 유효한 값인지 모름)

## 7. 원인 찾기지역마다 기준이 다름.**원인 1** : 생략**원인 2** : 지역 offset| 지역 | 활엽 | 침엽 | 두 클래스의 경계 ||---|---|---|---|| 홍천 | 0.369 | 0.538 | **0.453** || 대전 | 0.440 | 0.655 | 0.548 || 순천 | 0.515 | 0.742 | **0.629** |**순천의 활엽수(0.515)가 홍천의 침엽수(0.538)와 거의 동일.**지역 간 차이(0.204)가 클래스 간 차이(0.169)보다 큼.즉 **"활엽이냐 침엽이냐"보다 "어느 지역이냐"가 더 중요.**→ 순천에서 배운 경계 0.629를 홍천(0.453)에 그대로 적용하면　 홍천 침엽수가 기준 미달로 활엽 처리됨. 이게 +5.92%.

## 8. 고치려고 해본 것| | 결과 ||---|---|| **1. 모델을 키워봄** | 효과 적음 || **2. 수종 추가** | 효과 적음 || **3. 지역별 offset 정규화** | **이건 유의미. 근데 보정값(α)을 찾아야 함** |

## 9. 지금까지 모르는 것1. **LCA 기준** — 탄소축적량을 LCA에 어떻게 쓰는지2. **오차 범위** — 탄소 오차 몇 %까지 괜찮은지

## 10. 해보고 있는 것**지역을 늘려서 α 값 찾기**---### α가 뭔가**정규화**는 각 지역의 값을 "그 지역 안에서의 상대 위치"로 바꾸는 것.각 지역 자기 중앙값을 빼고 자기 표준편차로 나눔.→ **위성 통계만 쓰므로 라벨이 필요 없음.**실제로 축은 정렬됐음. 지역 간 차이가 클래스 간 차이 대비 **0.86 → 0.31**로 줄어듦.**그런데 6조합 전부 부호가 반전됨.**| | 정규화 전 → 후 ||---|---|| 순천 → 홍천 | +4.20 → **−2.03** || 홍천 → 대전 | −1.11 → **+1.94** |0을 향해 가다가 **멈추지 못하고 반대편으로 넘어감.** 평균 **1.9배 과보정.****왜 과보정인가**정규화는 라벨 없이 **전체 픽셀 중앙값**을 뺌. 그런데 그 안에 두 가지가 섞여 있음.| 빼야 할 것 | 빼면 안 되는 것 ||---|---|| **환경 차이** — 같은 소나무가 홍천 0.626, 순천 0.760 | **구성 차이** — 순천은 침엽이 47.6%라 중앙값이 위로 끌림 || 잡음이므로 제거가 맞음 | "순천에 침엽이 많다"는 **사실.** 모델이 알아야 할 정보 |순천 전체 중앙값 0.657, 클래스 경계 0.629. **0.028만큼 더 뺌.**→ **방향은 맞았고 세기가 과했음.****그래서 α**정규화 세기를 조절하는 계수. **0이면 정규화 안 함, 1이면 전량 적용.**| α | 평균 탄소 오차 ||---|---|| 0 (안 함) | 2.19%p || 1 (전량) | 1.74%p || **0.5 (절반)** | **0.49%p** (추정) |*※ 두 점만 알고 중간을 직선으로 채운 추정치. 실제로 훑어봐야 함*---### 왜 지역을 늘리나**α를 찾을 때와 쓸 때가 다름.**| | α 찾기 | α 쓰기 ||---|---|---|| 언제 | 연구 단계, **1회** | 실무, 매번 || 라벨 | **필요** | **불필요** || 어디서 | 지역 5~10개 | 새 지역 어디든 |α를 맞출 때는 정답이 있어야 채점이 됨. **임상도는 전국에 있으니 가능.**α가 정해지면 새 지역엔 **위성영상만** 넣으면 됨. 중앙값·표준편차는 영상에서 바로 나옴.> 체중계 영점 맞추기와 같음. **맞출 땐 무게를 아는 추가 필요하지만, 한 번 맞춰놓으면 그냥 올라가면 됨.****주의**: 여러 지역 데이터를 **합쳐서 모델 하나를 만드는 게 아님.**(그건 이미 해봤고 실패 — 대전+순천 학습이 대전 단독보다 홍천에서 나빴음)모델은 지역별로 각자 학습하고, **보정 규칙 α만 공유**하는 것.---### 순서| 순 | 할 일 | 시간 | 확인할 것 ||---|---|---|---|| **1** | 3지역에서 α 0~1 스캔 | 2시간 | 곡선이 0을 지나는가 || 2 | 평가셋 공간 분리로 수정 | 45분 | 5절 대각선 편향 제거 || 3 | 지역 3~4개 추가 | 지역당 반나절 | α가 좁게 모이는가 |**1번을 먼저.** 데이터를 새로 안 만들어도 됨(기존 샘플 캐시 사용).여기서 α 개념이 성립 안 하면 지역을 늘려도 소용없음.**지역 선정 시 맞춰야 할 조건**- 갱신년도 2024+ (정답이 낡으면 채점이 틀림)- **산림 비율 높은 곳** — 서울·부산 같은 도시는 파편화가 심해 경계 픽셀 오차가 α 추정을 오염시킴- 평가 방식·학습 픽셀 수 동일하나라도 안 맞으면, α가 흩어졌을 때 **"지역이 진짜 달라서"인지 "조건이 안 맞아서"인지 구분 불가.**---### 잘 되면지금은 **"전이가 이렇게 깨진다"는 진단**.α가 안정적으로 존재하면 **"이렇게 고친다"는 처방**이 붙음.그것도 **라벨 없이 적용 가능한 처방.****안 될 수도 있음.** 지금 추정으로 α가 0.36~0.79로 흩어져 있어하나로 모일 거라는 보장은 아직 없음.